In [128]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import scanpy as sc
from starcat import starCAT
from scipy.stats import ttest_ind, fisher_exact, ranksums, wilcoxon
import sys
sys.path.append('../../../Code/')
from utils import read_dataset_log

In [2]:
!pwd

/data/srlab1/TCAT/Analysis/Revision/Melanoma_Outcome


# Download data

In [152]:
! mkdir -p ../../../Data/Revision
! mkdir -p ../../../Data/Revision/Melanoma_Clinical_GSE120575

In [153]:
! wget -O ../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz https://ftp.ncbi.nlm.nih.gov/geo/series/GSE120nnn/GSE120575/suppl/GSE120575%5FSade%5FFeldman%5Fmelanoma%5Fsingle%5Fcells%5FTPM%5FGEO.txt.gz

--2024-09-05 23:27:03--  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE120nnn/GSE120575/suppl/GSE120575%5FSade%5FFeldman%5Fmelanoma%5Fsingle%5Fcells%5FTPM%5FGEO.txt.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 130.14.250.10, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126721504 (121M) [application/x-gzip]
Saving to: ‘../../../Data/Revision/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz’

100%[======================================>] 126,721,504 8.45MB/s   in 15s    

2024-09-05 23:27:18 (8.20 MB/s) - ‘../../../Data/Revision/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz’ saved [126721504/126721504]



In [154]:
! gzip -d ../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz

In [155]:
! wget -O ../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_patient_ID_single_cells.txt.gz https://ftp.ncbi.nlm.nih.gov/geo/series/GSE120nnn/GSE120575/suppl/GSE120575%5Fpatient%5FID%5Fsingle%5Fcells.txt.gz

--2024-09-05 23:28:04--  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE120nnn/GSE120575/suppl/GSE120575%5Fpatient%5FID%5Fsingle%5Fcells.txt.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.10, 130.14.250.11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 83035 (81K) [application/x-gzip]
Saving to: ‘../../../Data/Revision/GSE120575_patient_ID_single_cells.txt.gz’

100%[======================================>] 83,035      --.-K/s   in 0.1s    

2024-09-05 23:28:04 (799 KB/s) - ‘../../../Data/Revision/GSE120575_patient_ID_single_cells.txt.gz’ saved [83035/83035]



In [156]:
! gzip -d ../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_patient_ID_single_cells.txt.gz

# Create adata file

In [157]:
count = 0
cell_label = []
with open('../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt') as F:
    while count < 2:
        cell_label.append(F.readline().rstrip().split('\t')[1:])
        count += 1

cell_label = pd.DataFrame(cell_label).T
cell_label.columns = ['cell_id', 'sample_label']
cell_label.index = cell_label['cell_id']
cell_label.head()

,cell_id,sample_label
cell_id,,
A10_P3_M11,A10_P3_M11,Pre_P1
A11_P1_M11,A11_P1_M11,Pre_P1
A11_P3_M11,A11_P3_M11,Pre_P1
A11_P4_M11,A11_P4_M11,Pre_P1
A12_P3_M11,A12_P3_M11,Pre_P1


In [158]:
cell_label.iloc[-1, :]

cell_id         H9_P5_M67_L001_T_enriched
sample_label           Post_P6_T_enriched
Name: H9_P5_M67_L001_T_enriched, dtype: object

In [159]:
data = pd.read_csv('../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt', sep='\t', skiprows=2, index_col=0, header=None)

In [160]:
data = data.loc[:,data.isnull().sum(axis=0)==0]

In [161]:
adata = sc.AnnData(X=sp.csr_matrix(data.T.values), var=pd.DataFrame(data.index, index=data.index, columns=['gene_symbol']),
           obs=cell_label)
adata

AnnData object with n_obs × n_vars = 16291 × 55737
    obs: 'cell_id', 'sample_label'
    var: 'gene_symbol'

In [163]:
obsdata = pd.read_csv('../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_patient_ID_single_cells.txt', sep='\t', skiprows=19, encoding='ISO-8859-1')
(~obsdata.isnull()).sum(axis=1).value_counts()

7    16291
1       14
2       11
0       10
3        1
6        1
4        1
Name: count, dtype: int64

In [164]:
obsdata = obsdata.loc[(~obsdata.isnull()).sum(axis=1)==7, :].iloc[:, :7]
obsdata.index = obsdata['title']
obsdata.head()

,Sample name,title,source name,organism,characteristics: patinet ID (Pre=baseline; Post= on treatment),characteristics: response,characteristics: therapy
title,,,,,,,
A10_P3_M11,Sample 1,A10_P3_M11,Melanoma single cell,Homo sapiens,Pre_P1,Responder,anti-CTLA4
A11_P1_M11,Sample 2,A11_P1_M11,Melanoma single cell,Homo sapiens,Pre_P1,Responder,anti-CTLA4
A11_P3_M11,Sample 3,A11_P3_M11,Melanoma single cell,Homo sapiens,Pre_P1,Responder,anti-CTLA4
A11_P4_M11,Sample 4,A11_P4_M11,Melanoma single cell,Homo sapiens,Pre_P1,Responder,anti-CTLA4
A12_P3_M11,Sample 5,A12_P3_M11,Melanoma single cell,Homo sapiens,Pre_P1,Responder,anti-CTLA4


In [165]:
adata.obs = obsdata.loc[adata.obs.index, :]

In [166]:
adata.obs['organism'].value_counts()

organism
Homo sapiens    16291
Name: count, dtype: int64

In [167]:
adata.obs['source name'].value_counts()

source name
Melanoma single cell    16291
Name: count, dtype: int64

In [168]:
adata.obs = adata.obs.drop(['organism', 'source name'], axis=1)

In [169]:
adata.obs = adata.obs.rename(columns={'characteristics: patinet ID (Pre=baseline; Post= on treatment)':'sample', 'Sample name':'cell_label', 'title':'cell_id',
                         'characteristics: response':'response', 'characteristics: therapy':'therapy'})

In [171]:
adata.obs.head()

,cell_label,cell_id,sample,response,therapy
cell_id,,,,,
A10_P3_M11,Sample 1,A10_P3_M11,Pre_P1,Responder,anti-CTLA4
A11_P1_M11,Sample 2,A11_P1_M11,Pre_P1,Responder,anti-CTLA4
A11_P3_M11,Sample 3,A11_P3_M11,Pre_P1,Responder,anti-CTLA4
A11_P4_M11,Sample 4,A11_P4_M11,Pre_P1,Responder,anti-CTLA4
A12_P3_M11,Sample 5,A12_P3_M11,Pre_P1,Responder,anti-CTLA4


In [174]:
adata.obs['timepoint'] = adata.obs['sample'].apply(lambda x: x.split('_')[0])

In [178]:
adata.obs['patient'] = adata.obs['sample'].apply(lambda x: x.split('_')[1])

In [179]:
adata.obs.head()

,cell_label,cell_id,sample,response,therapy,timepoint,patient
cell_id,,,,,,,
A10_P3_M11,Sample 1,A10_P3_M11,Pre_P1,Responder,anti-CTLA4,Pre,P1
A11_P1_M11,Sample 2,A11_P1_M11,Pre_P1,Responder,anti-CTLA4,Pre,P1
A11_P3_M11,Sample 3,A11_P3_M11,Pre_P1,Responder,anti-CTLA4,Pre,P1
A11_P4_M11,Sample 4,A11_P4_M11,Pre_P1,Responder,anti-CTLA4,Pre,P1
A12_P3_M11,Sample 5,A12_P3_M11,Pre_P1,Responder,anti-CTLA4,Pre,P1


In [183]:
adata.var['gene_symbol'] = adata.var.index

In [184]:
adata.var.head()

,gene_symbol
0,
TSPAN6,TSPAN6
TNMD,TNMD
DPM1,DPM1
SCYL3,SCYL3
C1orf112,C1orf112


In [189]:
adata.var.index = adata.var.index.values

In [190]:
adata.var.head()

,gene_symbol
TSPAN6,TSPAN6
TNMD,TNMD
DPM1,DPM1
SCYL3,SCYL3
C1orf112,C1orf112


In [191]:
adata.var.isnull().sum()

gene_symbol    0
dtype: int64

In [219]:
clusterdat = pd.ExcelFile('../../../Data/Revision/Melanoma_Clinical_GSE120575/mmc1.xlsx').parse('Cluster annotation-Fig1B-C')
clusterdat['Cell Name Orig'] = clusterdat['Cell Name'].copy()
clusterdat['Cell Name'] = clusterdat['Cell Name'].apply(lambda x: x.replace('_DP1', '_enriched'))
clusterdat['Cell Name'] = clusterdat['Cell Name'].apply(lambda x: x.replace('_DP', '_enriched'))
clusterdat['Cell Name'] = clusterdat['Cell Name'].apply(lambda x: x.replace('_DN1', '_enriched'))
clusterdat['Cell Name'] = clusterdat['Cell Name'].apply(lambda x: x.replace('_DN', '_enriched'))
clusterdat.head()

/PHShome/dk718/miniforge3/envs/py310/lib/python3.10/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


,Cell Name,Cluster number,Cell Name Orig
0,A10_P3_M11,5,A10_P3_M11
1,A11_P1_M11,5,A11_P1_M11
2,A11_P3_M11,5,A11_P3_M11
3,A11_P4_M11,7,A11_P4_M11
4,A12_P3_M11,5,A12_P3_M11


In [220]:
adata = sc.read('../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_melanoma_scRNA.h5ad')
adata.obs.index = np.array(adata.obs.index)

In [221]:
adata.obs['cell_id_orig'] = adata.obs['cell_id'].copy()
adata.obs['cell_id'] = adata.obs['cell_id'].apply(lambda x: x.replace('_T_enriched', '_enriched'))
adata.obs['cell_id'] = adata.obs['cell_id'].apply(lambda x: x.replace('_myeloid_enriched', '_enriched'))


#ind = (adata.obs['cell_id'].apply(lambda x: 'enriched' in x)).values
#adata.obs.loc[ind, 'cell_id'] = adata.obs.loc[ind, 'cell_id'].apply(lambda x: '_'.join(x.split('_')[:-2]))
#adata.obs.index = adata.obs['cell_id']

In [222]:
x = sorted(set(adata.obs['cell_id']) - set(clusterdat['Cell Name']))
y = sorted(set(clusterdat['Cell Name']) - set(adata.obs['cell_id']))
print(len(x), len(y))

304 304


In [225]:
ind = ~adata.obs['cell_id'].isin(clusterdat['Cell Name']) & adata.obs['cell_id'].apply(lambda x: x.replace('_enriched', '')).isin(clusterdat['Cell Name'])
adata.obs.loc[ind, 'cell_id'] = adata.obs.loc[ind, 'cell_id'].apply(lambda x: x.replace('_enriched', ''))

In [226]:
x = sorted(set(adata.obs['cell_id']) - set(clusterdat['Cell Name']))
y = sorted(set(clusterdat['Cell Name']) - set(adata.obs['cell_id']))
print(len(x), len(y))

0 0


In [227]:
adata.obs['cell_id'].value_counts().head()

cell_id
H9_P5_M67_L001    1
A10_P3_M11        1
A11_P1_M11        1
E6_P5_M67_L001    1
E7_P5_M67_L001    1
Name: count, dtype: int64

In [228]:
clusterdat['Cell Name'].value_counts().head()

Cell Name
H9_P5_M67_L001    1
A10_P3_M11        1
A11_P1_M11        1
E6_P5_M67_L001    1
E7_P5_M67_L001    1
Name: count, dtype: int64

In [229]:
merged = pd.merge(left=adata.obs, right=clusterdat, left_on='cell_id', right_on='Cell Name', how='left')

In [232]:
merged.index = merged['cell_id'].values

In [234]:
adata.obs = merged.loc[adata.obs['cell_id'], :]

In [239]:
adata.obs['CD8+CD39+TIM+_Sorted'] = False
ind = adata.obs['Cell Name Orig'].apply(lambda x: '_DP' in x)
adata.obs.loc[ind, 'CD8+CD39+TIM+_Sorted'] = True

adata.obs['CD8+CD39-TIM-_Sorted'] = False
ind = adata.obs['Cell Name Orig'].apply(lambda x: '_DN' in x)
adata.obs.loc[ind, 'CD8+CD39-TIM-_Sorted'] = True

In [241]:
adata.obs.loc[adata.obs['Cell Name Orig'].apply(lambda x: '_DN' in x), :]

,cell_label,cell_id,sample,response,therapy,timepoint,patient,cell_id_orig,Cell Name,Cluster number,Cell Name Orig,CD8+CD39+TIM+_Sorted,CD8+CD39-TIM-_Sorted
A10_P1_M39_enriched,Sample 15419,A10_P1_M39_enriched,Post_P2,Non-responder,anti-PD1,Post,P2,A10_P1_M39_T_enriched,A10_P1_M39_enriched,8,A10_P1_M39_DN1,False,True
A11_P1_M39_enriched,Sample 15420,A11_P1_M39_enriched,Post_P2,Non-responder,anti-PD1,Post,P2,A11_P1_M39_T_enriched,A11_P1_M39_enriched,8,A11_P1_M39_DN1,False,True
A12_P1_M39_enriched,Sample 15421,A12_P1_M39_enriched,Post_P2,Non-responder,anti-PD1,Post,P2,A12_P1_M39_T_enriched,A12_P1_M39_enriched,5,A12_P1_M39_DN1,False,True
A1_P1_M39_enriched,Sample 15422,A1_P1_M39_enriched,Post_P2,Non-responder,anti-PD1,Post,P2,A1_P1_M39_T_enriched,A1_P1_M39_enriched,5,A1_P1_M39_DN1,False,True
A3_P1_M39_enriched,Sample 15423,A3_P1_M39_enriched,Post_P2,Non-responder,anti-PD1,Post,P2,A3_P1_M39_T_enriched,A3_P1_M39_enriched,8,A3_P1_M39_DN1,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
H5_P8_MMD5_enriched,Sample 16101,H5_P8_MMD5_enriched,Post_P8,Responder,anti-CTLA4+PD1,Post,P8,H5_P8_MMD5_T_enriched,H5_P8_MMD5_enriched,6,H5_P8_MMD5_DN,False,True
H6_P8_MMD5_enriched,Sample 16102,H6_P8_MMD5_enriched,Post_P8,Responder,anti-CTLA4+PD1,Post,P8,H6_P8_MMD5_T_enriched,H6_P8_MMD5_enriched,8,H6_P8_MMD5_DN,False,True
H7_P8_MMD5_enriched,Sample 16103,H7_P8_MMD5_enriched,Post_P8,Responder,anti-CTLA4+PD1,Post,P8,H7_P8_MMD5_T_enriched,H7_P8_MMD5_enriched,6,H7_P8_MMD5_DN,False,True
H8_P8_MMD5_enriched,Sample 16104,H8_P8_MMD5_enriched,Post_P8,Responder,anti-CTLA4+PD1,Post,P8,H8_P8_MMD5_T_enriched,H8_P8_MMD5_enriched,6,H8_P8_MMD5_DN,False,True


In [243]:
cluster_map = {1:'B',
               2:'Plasma',
               3:'Mono/Mac',
               4:'Dendritic',
               5:'Lymphocytes',
               6:'Exhausted CD8',
               7:'Treg',
               8:'Cytotoxicity',
               9:'Exhausted/HS CD8',
               10:'Memory T',
               11:'Lymphocyte exhausted/cell cycle'}

In [246]:
adata.obs['Cluster label'] = adata.obs['Cluster number'].replace(cluster_map)

In [249]:
adata.obs['Cluster label'].value_counts()

Cluster label
Exhausted CD8                      2222
Lymphocytes                        2165
Cytotoxicity                       2165
Memory T                           1773
Treg                               1740
Exhausted/HS CD8                   1656
B                                  1455
Mono/Mac                           1391
Lymphocyte exhausted/cell cycle    1129
Plasma                              305
Dendritic                           290
Name: count, dtype: int64

In [247]:
adata.write('../../../Data/Revision/Melanoma_Clinical_GSE120575/GSE120575_melanoma_scRNA.h5ad')